In [ ]:
#load dataset
import pandas as pd
import numpy as np
import os

data_path = "data/raw/"

customers = pd.read_csv(
    data_path + "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    data_path + "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    data_path + "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    data_path + "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    data_path + "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    data_path + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    data_path + "olist_sellers_dataset.csv"
)

geolocation = pd.read_csv(
    data_path + "olist_geolocation_dataset.csv"
)

translation = pd.read_csv(
    data_path + "product_category_name_translation.csv"
)

In [4]:
#standardize column names

datasets = [
    customers,
    orders,
    order_items,
    payments,
    reviews,
    products,
    sellers,
    geolocation,
    translation
]

for df in datasets:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [9]:
#convert date columns to datetime format

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

    order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

delivery_days = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [10]:
#handle product category translation

products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)


products[
    "product_category_name_english"
] = products[
    "product_category_name_english"
].fillna("unknown")

In [11]:
#create revenue feature

order_items["item_revenue"] = (
    order_items["price"]
    + order_items["freight_value"]
)

In [12]:
#create order level metrics

order_metrics = (
    order_items
    .groupby("order_id")
    .agg(
        item_revenue=("item_revenue", "sum"),
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)


orders = orders.merge(
    order_metrics,
    on="order_id",
    how="left"
)

In [13]:
#create time features

orders["order_year"] = (
    orders["order_purchase_timestamp"].dt.year
)

orders["order_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [14]:
#create delivery metrices

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [15]:
#create delivery status

orders["is_delivered"] = (
    orders["order_status"] == "delivered"
)

orders["is_canceled"] = (
    orders["order_status"] == "canceled"
)

orders["is_unavailable"] = (
    orders["order_status"] == "unavailable"
)

orders["delivery_late"] = np.where(
    orders["is_delivered"]
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna(),

    orders["order_delivered_customer_date"]
    > orders["order_estimated_delivery_date"],

    np.nan
)

In [16]:
#customer repeat analysis
customer_orders = orders.merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

#no. of order per customer
frequency = (
    customer_orders
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

customer_orders["customer_order_count"] = (
    customer_orders["customer_unique_id"]
    .map(frequency)
)

customer_orders["is_repeat_customer"] = (
    customer_orders["customer_order_count"] > 1
)

In [17]:
#Don't "fix" suspicious dates instead create flags

orders["invalid_carrier_sequence"] = (
    orders["order_delivered_carrier_date"].notna()
    & orders["order_approved_at"].notna()
    & (
        orders["order_delivered_carrier_date"]
        < orders["order_approved_at"]
    )
)

orders["invalid_delivery_sequence"] = (
    orders["order_delivered_customer_date"].notna()
    & orders["order_delivered_carrier_date"].notna()
    & (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    )
)

In [ ]:
import os

# Create processed folder
os.makedirs("../data/processed", exist_ok=True)

# Save cleaned datasets
customers.to_csv("../data/processed/customers_clean.csv", index=False)
orders.to_csv("../data/processed/orders_clean.csv", index=False)
order_items.to_csv("../data/processed/order_items_clean.csv", index=False)
payments.to_csv("../data/processed/order_payments_clean.csv", index=False)
reviews.to_csv("../data/processed/order_reviews_clean.csv", index=False)
products.to_csv("../data/processed/products_clean.csv", index=False)
sellers.to_csv("../data/processed/sellers_clean.csv", index=False)
geolocation.to_csv("../data/processed/geolocation_clean.csv", index=False)
translation.to_csv("../data/processed/category_translation.csv", index=False)

print("All processed files saved successfully!")